In [ ]:
import os
import shutil
import requests
from email.utils import parsedate_to_datetime
from zoneinfo import ZoneInfo

URL = "https://opendata.transport.vic.gov.au/dataset/3f4e292e-7f8a-4ffe-831f-1953be0fe448/resource/fb152201-859f-4882-9206-b768060b50ad/download/gtfs.zip"

# Catalog is passed as a job parameter (default -> ${var.catalog} per target); the default here
# is only a convenience for interactive runs.
dbutils.widgets.text("catalog", "transport_vic_dev")
CATALOG = dbutils.widgets.get("catalog")

# Last-Modified is the feed's publication time. It's the only version key the endpoint gives us,
# so bail loudly rather than silently partitioning under the wrong date.
head = requests.head(URL, timeout=30, allow_redirects=True)
head.raise_for_status()
last_modified = head.headers.get("Last-Modified")
if last_modified is None:
    raise RuntimeError(f"No Last-Modified header on {URL}; cannot derive export_date.")

# parsedate_to_datetime returns UTC; the feed is published on Melbourne time, so convert before
# taking the date or a late-evening AEST export lands in the previous day's partition.
export_date = (
    parsedate_to_datetime(last_modified)
    .astimezone(ZoneInfo("Australia/Melbourne"))
    .strftime("%Y%m%d")
)

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.`01_bronze`.gtfs_raw")
dest = f"/Volumes/{CATALOG}/01_bronze/gtfs_raw/export_date={export_date}/gtfs.zip"

if os.path.exists(dest):
    # Same export already landed — the feed only rolls weekly, so re-running is a no-op.
    print(f"{dest} already exists; skipping download.")
else:
    # Volumes don't support non-sequential writes, so stage on local ephemeral disk and copy in.
    # The copy is the last step, so a failed download never leaves a partial file under dest.
    local_tmp = f"/tmp/gtfs_{export_date}.zip"
    with requests.get(URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(local_tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)

    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copyfile(local_tmp, dest)
    os.remove(local_tmp)
    print(f"Downloaded {os.path.getsize(dest)} bytes -> {dest}")

# Downstream tasks read this to locate the export they should unzip/parse.
dbutils.jobs.taskValues.set("export_date", export_date)
